# Journey to Auto Merge

## Why This Journey Mattered

The core frustration was simple: notebooks lose important state when the session dies. A long-running job finishes, outputs exist only in memory, and then the kernel or process disappears. For me, the central problem was never just multiplayer editing. It was making sure users did not lose their work, especially their outputs and the computational state that gave those outputs meaning.

That framing matters because it changes what counts as success. A collaboration system can be technically impressive and still miss the practical need. I was less interested in people typing in the same notebook at the same time than in building a system where notebooks, runtimes, and eventually agents could participate as peers without dropping the results of computation.

That is also why the notebook itself started to feel like such an important substrate. Agents already thrive in a REPL loop, but they are usually missing durable state. A notebook can provide that missing layer: persistent state across cells, visible outputs, and a shared place where humans and agents can work against the same evolving artifact.

## The Evolution

The journey to **Automerge** followed a chronological progression through different state management paradigms, but the thread running through all of them was the same: how do we make notebook state durable, distributed, and usable by more than a single UI process?

1. **Redux** — A deterministic mental model where state is a function of prior state and an event
2. **Distributed Redux** — Extending that model across machines with deltas that could be squashed into current state
3. **Livestore** — Treating events as truth and deriving materialized state from an event log
4. **Automerge & YJS** — Exploring CRDTs and modern collaboration primitives
5. **Back to Automerge** — Choosing the model that best fit notebooks, runtimes, and agent peers

### 1. Redux

Redux felt right early on because it offered a pure and deterministic way to think about state: the next state is a function of the current state and an event. That mental model was clean, predictable, and easy to reason about. It matched how many of us wanted to build software.

We pushed that style pretty far, including with things like Redux Observable and RxJS, because events felt like the right primitive. In many ways, they were. But that model also carries implicit ordering assumptions. It works best when there is a clear sequence of actions and a relatively centralized authority over how those actions are applied.

### 2. Distributed Redux

The next step was to ask whether the Redux mental model could survive distribution. At Notable, we built a system that carried deltas around the notebook so different participants could apply them and squash them into the current state. That let server-side code, Python processes, or really any language runtime consume the same stream of changes and converge on a usable representation.

There was a lot to like in that approach. It preserved some of the lessons from Redux while acknowledging that notebook state lives in more than one place. But it also became harder to extend into a truly multi-actor world. Events needed attribution. Ordering needed to be defended. You had to trust that a stream of decorated deltas really came from the entities they claimed to come from. The model still wanted a stronger notion of sequence and authority than the problem space naturally provided.

Around the same time, Jupyter's collaboration work was focused heavily on realtime multiplayer editing. That was technically solid, but it was not quite the problem I most wanted to solve. My goal was not primarily human multiplayer. My goal was to make sure users did not lose outputs.

### 3. Livestore

Livestore shifted the frame again by embracing an event-sourced model. Events became the durable truth, and materialized state could be derived from them, even into something like SQLite. That had real appeal. SQLite is a fantastic substrate: portable, dependable, and increasingly available everywhere, including the browser via Wasm.

The developer ergonomics were strong too. You could write events, derive state, and think clearly about transitions over time. In that sense, Livestore felt like a more principled continuation of the event-driven path.

But there were still gaps for the kind of system I wanted. Compaction becomes necessary. Server-side participation matters. Runtime agents need to be peers that can write to the shared document too, not just consumers of a frontend-oriented collaboration model. Livestore was compelling, but it still was not the full answer for notebook computation, persistence, and multi-actor participation.

### 4. Automerge & YJS

CRDTs were the next obvious place to look. I went back and forth between Automerge and YJS because both offered a way to build shared state without relying on a single fragile ordering story. They gave collaboration primitives that were more native to distribution.

But even here, the real evaluation criterion was not just whether multiple humans could edit the same notebook. The question was whether the model could support notebooks as computational artifacts: cells, outputs, metadata, runtimes, and eventually agents all participating in the same shared state.

In projects like nteract desktop, that need realtime sync across windows and agents, Automerge became more than a theoretical interest. It offered a concrete way to treat shared state as a canonical live document in the daemon while still merging back into notebook files and preserving metadata that the CRDT layer does not own.

### 5. Back to Automerge

The return to Automerge was not really a return to collaboration for collaboration's sake. It was a return to a data model that could better match the actual problem. Notebooks are not just text editors. They are living computational artifacts with cells, outputs, metadata, runtimes, and now agents.

Automerge fit because it made it possible to treat shared state as a real document with peers that do not have to be privileged in the old centralized way. In systems like nteract desktop, that means a daemon can hold the canonical live document, windows can maintain replicas, and the notebook can still be merged back to `.ipynb` while preserving metadata that the shared document does not own.

That is much closer to the goal that started this whole journey: users should not lose their outputs, and computation itself should be able to participate in the same shared state model as the UI.

## What This Implies

Following this path changed how I think about notebook architecture. The kernel, the UI, and the coordinating server should be decoupled. The notebook should not be a fragile front-end container for state that disappears when a tab closes. It should be a durable shared document that computation can keep updating even when the interface goes away for a while.

That is why this journey keeps circling back to sync engines. The sync model is not an implementation detail. It determines whether notebooks can support long-running work, preserve outputs, and make room for agents as first-class collaborators instead of bolted-on automation.